In [16]:
import numpy as np

def load_rtl_layer(path, nfrac):
    raw = np.loadtxt(path, delimiter=',')   # shape (n_positions, n_channels)
    flat = raw.flatten()                     # row-major: channel varies fastest, matches (H,W,C) flatten
    return flat / (2 ** nfrac)

def load_ref_layer(path):
    return np.loadtxt(path, delimiter=',')   # already flat, already decimal

rtl_conv0 = load_rtl_layer("traces/rtl_0_conv0.csv", nfrac=6)
hls4ml_conv0 = load_ref_layer("traces/hls4ml_0_q_activation.csv")

print(rtl_conv0.shape, hls4ml_conv0.shape)  # MUST match before comparing values

diff = np.abs(rtl_conv0 - hls4ml_conv0)
print("mean:", diff.mean())
print("median:", np.median(diff))
print("p95:", np.percentile(diff, 95))
print("max:", diff.max())
print(f"num outliers (>0.1): {(diff > 0.1).sum()} / {diff.size}")

(12696,) (12696,)
mean: 2.4614051669817266e-06
median: 0.0
p95: 0.0
max: 0.015625
num outliers (>0.1): 0 / 12696


In [17]:
print("rtl:   ", rtl_conv0[:12])
print("hls4ml:", hls4ml_conv0[:12])

rtl:    [0.       0.265625 0.359375 0.       0.3125   0.       0.       0.
 0.       0.       0.       0.      ]
hls4ml: [0.      0.25    0.34375 0.      0.3125  0.      0.      0.      0.
 0.      0.      0.     ]


In [18]:
outlier_idx = np.where(diff > 0.1)[0]
print("outlier indices:", outlier_idx)
print("rtl values:", rtl_conv0[outlier_idx])
print("hls4ml values:", hls4ml_conv0[outlier_idx])

outlier indices: []
rtl values: []
hls4ml values: []


In [19]:
rtl_2d = np.loadtxt("traces/rtl_0_conv0.csv", delimiter=',')  # shape (2116, 6)
# check which column(s) most often have this negative-not-zeroed pattern
for ch in range(6):
    neg_count = (rtl_2d[:, ch] < 0).sum()
    print(f"channel {ch}: {neg_count} negative raw values out of {rtl_2d.shape[0]}")

channel 0: 0 negative raw values out of 2116
channel 1: 0 negative raw values out of 2116
channel 2: 0 negative raw values out of 2116
channel 3: 0 negative raw values out of 2116
channel 4: 0 negative raw values out of 2116
channel 5: 0 negative raw values out of 2116


In [20]:
def diff_stats(rtl_path, ref_path, nfrac):
    rtl = load_rtl_layer(rtl_path, nfrac)
    ref = load_ref_layer(ref_path)
    diff = np.abs(rtl - ref)
    print(f"{ref_path:45s} mean={diff.mean():.6f} median={np.median(diff):.6f} p95={np.percentile(diff,95):.6f} max={diff.max():.6f} outliers={int((diff>0.1).sum())}/{diff.size}")

diff_stats("traces/rtl_0_conv0.csv", "traces/hls4ml_0_q_activation.csv", nfrac=6)
diff_stats("traces/rtl_0_pool0.csv", "traces/hls4ml_0_max_pooling2d.csv", nfrac=6)
diff_stats("traces/rtl_0_conv1.csv", "traces/hls4ml_0_q_activation_1.csv", nfrac=6)
diff_stats("traces/rtl_0_pool1.csv", "traces/hls4ml_0_max_pooling2d_1.csv", nfrac=6)
diff_stats("traces/rtl_0_conv2.csv", "traces/hls4ml_0_q_activation_2.csv", nfrac=6)
diff_stats("traces/rtl_0_pool2.csv", "traces/hls4ml_0_max_pooling2d_2.csv", nfrac=6)
diff_stats("traces/rtl_0_dense0.csv", "traces/hls4ml_0_q_dense.csv", nfrac=5)
diff_stats("traces/rtl_0_relu0.csv", "traces/hls4ml_0_q_activation_3.csv", nfrac=5)
diff_stats("traces/rtl_0_dense1.csv", "traces/hls4ml_0_q_dense_1.csv", nfrac=5)
diff_stats("traces/rtl_0_relu1.csv", "traces/hls4ml_0_q_activation_4.csv", nfrac=5)
diff_stats("traces/rtl_0_final.csv", "traces/hls4ml_0_q_dense_2.csv", nfrac=5)

traces/hls4ml_0_q_activation.csv              mean=0.000002 median=0.000000 p95=0.000000 max=0.015625 outliers=0/12696
traces/hls4ml_0_max_pooling2d.csv             mean=0.000043 median=0.000000 p95=0.000000 max=0.015625 outliers=0/726
traces/hls4ml_0_q_activation_1.csv            mean=0.007957 median=0.000000 p95=0.062500 max=0.062500 outliers=0/648
traces/hls4ml_0_max_pooling2d_1.csv           mean=0.008545 median=0.000000 p95=0.062500 max=0.062500 outliers=0/128
traces/hls4ml_0_q_activation_2.csv            mean=0.030859 median=0.000000 p95=0.296875 max=0.343750 outliers=4/40
traces/hls4ml_0_max_pooling2d_2.csv           mean=0.029687 median=0.000000 p95=0.163281 max=0.296875 outliers=1/10
traces/hls4ml_0_q_dense.csv                   mean=0.010417 median=0.000000 p95=0.031250 max=0.031250 outliers=0/15
traces/hls4ml_0_q_activation_3.csv            mean=0.002083 median=0.000000 p95=0.009375 max=0.031250 outliers=0/15
traces/hls4ml_0_q_dense_1.csv                 mean=0.021875 median

In [21]:
rtl = load_rtl_layer("traces/rtl_0_conv2.csv", nfrac=6)
ref = load_ref_layer("traces/hls4ml_0_q_activation_2.csv")
diff = np.abs(rtl - ref)
outlier_idx = np.where(diff > 0.1)[0]
print("outlier indices:", outlier_idx, "-> channels:", outlier_idx % 10)  # conv2 has 10 channels
print("rtl:", rtl[outlier_idx])
print("hls4ml:", ref[outlier_idx])

outlier indices: [ 2 12 22 32] -> channels: [2 2 2 2]
rtl: [0.28125  0.328125 0.328125 0.328125]
hls4ml: [0.625 0.625 0.625 0.625]


In [22]:
def diff_stats(rtl_path, ref_path, nfrac):
    rtl = load_rtl_layer(rtl_path, nfrac)
    ref = load_ref_layer(ref_path)
    diff = np.abs(rtl - ref)
    print(f"{ref_path:42s} mean={diff.mean():.6f} median={np.median(diff):.6f} p95={np.percentile(diff,95):.6f} max={diff.max():.6f} outliers={int((diff>0.1).sum())}/{diff.size}")

# conv0 -- already validated, re-run to confirm the fix held
diff_stats("traces/rtl_0_conv0.csv", "traces/hls4ml_0_q_activation.csv", nfrac=6)

# pool0 -- no MAC, so scale should just pass through conv0's output scale
diff_stats("traces/rtl_0_pool0.csv", "traces/hls4ml_0_max_pooling2d.csv", nfrac=6)

# conv1 -- compare post-ReLU, same as conv0
diff_stats("traces/rtl_0_conv1.csv", "traces/hls4ml_0_q_activation_1.csv", nfrac=6)

# pool1
diff_stats("traces/rtl_0_pool1.csv", "traces/hls4ml_0_max_pooling2d_1.csv", nfrac=6)

# conv2 -- this is where you saw the channel-2 outlier before the ReLU fix; recheck now
diff_stats("traces/rtl_0_conv2.csv", "traces/hls4ml_0_q_activation_2.csv", nfrac=6)

# pool2
diff_stats("traces/rtl_0_pool2.csv", "traces/hls4ml_0_max_pooling2d_2.csv", nfrac=6)

# dense0/relu0 -- dense doesn't fuse ReLU (you have separate relu0 trace), so compare dense0 pre-activation directly
diff_stats("traces/rtl_0_dense0.csv", "traces/hls4ml_0_q_dense.csv", nfrac=5)
diff_stats("traces/rtl_0_relu0.csv", "traces/hls4ml_0_q_activation_3.csv", nfrac=5)

# dense1/relu1
diff_stats("traces/rtl_0_dense1.csv", "traces/hls4ml_0_q_dense_1.csv", nfrac=5)
diff_stats("traces/rtl_0_relu1.csv", "traces/hls4ml_0_q_activation_4.csv", nfrac=5)

traces/hls4ml_0_q_activation.csv           mean=0.000002 median=0.000000 p95=0.000000 max=0.015625 outliers=0/12696
traces/hls4ml_0_max_pooling2d.csv          mean=0.000043 median=0.000000 p95=0.000000 max=0.015625 outliers=0/726
traces/hls4ml_0_q_activation_1.csv         mean=0.007957 median=0.000000 p95=0.062500 max=0.062500 outliers=0/648
traces/hls4ml_0_max_pooling2d_1.csv        mean=0.008545 median=0.000000 p95=0.062500 max=0.062500 outliers=0/128
traces/hls4ml_0_q_activation_2.csv         mean=0.030859 median=0.000000 p95=0.296875 max=0.343750 outliers=4/40
traces/hls4ml_0_max_pooling2d_2.csv        mean=0.029687 median=0.000000 p95=0.163281 max=0.296875 outliers=1/10
traces/hls4ml_0_q_dense.csv                mean=0.010417 median=0.000000 p95=0.031250 max=0.031250 outliers=0/15
traces/hls4ml_0_q_activation_3.csv         mean=0.002083 median=0.000000 p95=0.009375 max=0.031250 outliers=0/15
traces/hls4ml_0_q_dense_1.csv              mean=0.021875 median=0.031250 p95=0.048437 max=